# twinfer on Colab

Builds the CUDA version, runs the tests, compares against Hugging Face transformers on the real
SmolLM2-135M-Instruct, and prints the benchmark tables for the README.

Switch the runtime to a GPU first (Runtime > Change runtime type > T4 GPU).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!nvcc --version | tail -2

In [ ]:
import os
os.chdir("/content")  # so running this cell again doesn't clone into the clone
if not os.path.exists("twinfer"):
    !git clone -q https://github.com/Yashvardhan0505/twinfer.git
%cd /content/twinfer
!git fetch -q && git reset -q --hard origin/main  # a scratch copy: always match GitHub

## Build and test

In [ ]:
!make -j2 cpu cuda build/test-common ARCH=native 2>&1 | tail -3
!./build/test-kernels

In [ ]:
!python3 tests/run_tests.py --cpu --cuda

## Real model vs Hugging Face

In [ ]:
from huggingface_hub import snapshot_download
snapshot_download("HuggingFaceTB/SmolLM2-135M-Instruct", local_dir="models/smollm2-135m",
                  allow_patterns=["*.json", "*.jinja", "*.safetensors"])
!python3 tools/hf_compare.py models/smollm2-135m --cuda

In [ ]:
!./build/twinfer-cuda -m models/smollm2-135m -p "Explain warp divergence to a beginner in three sentences." -n 200

## Throughput (paste this into the README)

In [ ]:
!python3 tools/bench_table.py models/smollm2-135m   # median of 3 runs per number

## Kernels vs cuBLAS, and the roofline

In [ ]:
!./build/bench-kernels --csv build/bench.csv | tee build/bench.md

In [ ]:
!python3 tools/roofline.py build/bench.csv -o build/roofline.png
from IPython.display import Image
Image("build/roofline.png")

In [ ]:
from google.colab import files
files.download("build/roofline.png")

The cell above downloads the chart. To update the README's chart, save it as `docs/roofline-t4.png` (or `roofline-<gpu>.png` for another GPU) in your repo and commit it.

To profile one decode step:

```
!nsys profile -o decode --force-overwrite true ./build/twinfer-cuda -m models/smollm2-135m --bench 32
!ncu --set full -k regex:gemv_kernel -c 5 -o gemv ./build/twinfer-cuda -m models/smollm2-135m --bench 4 --no-graph
```